# Análise de Churn — BankChurners

Notebook organizado em etapas: carga e limpeza dos dados, análise exploratória, treinamento de um modelo de Regressão Logística e avaliação do impacto financeiro das previsões.

## 1. Importação de bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")


## 2. Carregamento e limpeza dos dados

Carrega o CSV, remove as colunas geradas automaticamente pelo classificador Naive Bayes (que não fazem parte dos dados originais) e cria a coluna alvo `Churn` a partir de `Attrition_Flag`.

In [ ]:
df = pd.read_csv('BankChurners.csv')

colunas_para_remover = [
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2'
]
df = df.drop(columns=colunas_para_remover)

df["Churn"] = df["Attrition_Flag"].apply(lambda x: 1 if x == "Attrited Customer" else 0)
df = df.drop(columns=["Attrition_Flag"])

df.head()


## 3. Análise exploratória (EDA)

Comparação de duas variáveis-chave entre clientes ativos (`Churn = 0`) e clientes que cancelaram (`Churn = 1`):
- Quantidade total de transações no ano
- Meses de inatividade no último ano

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Relação entre o Churn e a Quantidade de Transações
sns.boxplot(x='Churn', y='Total_Trans_Ct', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Quantidade Total de Transações vs Churn')
ax[0].set_xlabel('0 = Cliente Ativo | 1 = Churn')
ax[0].set_ylabel('Total de Transações no Ano')

# Gráfico 2: Relação entre o Churn e os Meses Inativos
sns.boxplot(x='Churn', y='Months_Inactive_12_mon', data=df, ax=ax[1], palette='Set2')
ax[1].set_title('Meses Inativos no Último Ano vs Churn')
ax[1].set_xlabel('0 = Cliente Ativo | 1 = Churn')
ax[1].set_ylabel('Meses Inativos')

plt.tight_layout()
plt.show()


## 4. Pré-processamento

Converte variáveis categóricas em dummies, separa features (`X`) e alvo (`y`), divide em treino/teste (80/20, estratificado) e padroniza as variáveis numéricas.

In [ ]:
df_model = pd.get_dummies(df, drop_first=True)

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 5. Treinamento do modelo

Regressão Logística com `class_weight='balanced'` para compensar o desbalanceamento entre clientes ativos e em churn.

In [ ]:
modelo = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
modelo.fit(X_train_scaled, y_train)

previsoes = modelo.predict(X_test_scaled)

print("Modelo treinado")


## 6. Avaliação do modelo

Matriz de confusão e relatório de classificação (precisão, recall e F1-score) para a classe de churn.

In [ ]:
matriz = confusion_matrix(y_test, previsoes)

plt.figure(figsize=(8, 6))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Previu: Ativo (0)', 'Previu: Churn (1)'],
            yticklabels=['Real: Ativo (0)', 'Real: Churn (1)'])

plt.title('Matriz de Confusão', pad=20, size=16)
plt.ylabel('Acontecimento Real')
plt.xlabel('Previsão do Algoritmo')
plt.show()

print("\n--- Relatório de Desempenho do Modelo ---")
print(classification_report(y_test, previsoes))


## 7. Impacto financeiro

Estima, com base em uma taxa de retenção de 2% sobre o volume transacionado, quanto foi retido graças ao modelo (verdadeiros positivos) e quanto foi perdido nos pontos cegos (falsos negativos).

In [ ]:
resultados = X_test.copy()
resultados['Real_Churn'] = y_test
resultados['Previsao_Churn'] = previsoes
resultados['Volume_Transacoes'] = df.loc[X_test.index, 'Total_Trans_Amt']

# Clientes que iriam sair e o modelo detectou a tempo
clientes_salvos = resultados[(resultados['Real_Churn'] == 1) & (resultados['Previsao_Churn'] == 1)]
volume_salvo = clientes_salvos['Volume_Transacoes'].sum()
lucro_retido = volume_salvo * 0.02

# Clientes que saíram e o modelo não detectou (pontos cegos)
clientes_perdidos = resultados[(resultados['Real_Churn'] == 1) & (resultados['Previsao_Churn'] == 0)]
volume_perdido = clientes_perdidos['Volume_Transacoes'].sum()
lucro_perdido = volume_perdido * 0.02

print("=== RELATÓRIO DE IMPACTO FINANCEIRO DO MODELO ===")
print(f"Volume Financeiro em Risco Detectado: $ {volume_salvo:,.2f}")
print(f"Lucro Retido (2% de taxa): $ {lucro_retido:,.2f}")
print("-" * 45)
print(f"Volume Financeiro Perdido (Pontos Cegos): $ {volume_perdido:,.2f}")
print(f"Lucro Perdido: $ {lucro_perdido:,.2f}")
